# DRF Class-Based & Generic Views

## Why Class-Based Views?

Class-based views separate HTTP method logic into dedicated methods (`get`, `post`, `put`, `delete`) and enable inheritance and reuse.

## APIView

```python
from rest_framework.views import APIView
from rest_framework.response import Response
from .models import Book
from .serializers import BookModelSerializer

class BookListAPIView(APIView):
    def get(self, request):
        books = Book.objects.all()
        return Response(BookModelSerializer(books, many=True).data)

    def post(self, request):
        s = BookModelSerializer(data=request.data)
        if s.is_valid():
            s.save()
            return Response(s.data, status=201)
        return Response(s.errors, status=400)

class BookDetailAPIView(APIView):
    def get_object(self, pk):
        try:
            return Book.objects.get(pk=pk)
        except Book.DoesNotExist:
            return None

    def get(self, request, pk):
        book = self.get_object(pk)
        if not book:
            return Response(status=404)
        return Response(BookModelSerializer(book).data)

    def put(self, request, pk):
        book = self.get_object(pk)
        if not book:
            return Response(status=404)
        s = BookModelSerializer(book, data=request.data)
        if s.is_valid():
            s.save()
            return Response(s.data)
        return Response(s.errors, status=400)

    def delete(self, request, pk):
        book = self.get_object(pk)
        if not book:
            return Response(status=404)
        book.delete()
        return Response(status=204)
```


## GenericAPIView

`GenericAPIView` adds helpers like `get_queryset()` and `get_serializer()`, reducing boilerplate:

```python
from rest_framework.generics import GenericAPIView
from rest_framework.response import Response

class BookListGenericView(GenericAPIView):
    queryset = Book.objects.all()
    serializer_class = BookModelSerializer

    def get(self, request):
        s = self.get_serializer(self.get_queryset(), many=True)
        return Response(s.data)

    def post(self, request):
        s = self.get_serializer(data=request.data)
        if s.is_valid():
            s.save()
            return Response(s.data, status=201)
        return Response(s.errors, status=400)
```


## Mixins

Mixins are pre-built behaviors that provide methods like `list()`, `create()`, `retrieve()`, `update()`, and `destroy()`.

```python
from rest_framework.mixins import ListModelMixin, CreateModelMixin, RetrieveModelMixin, UpdateModelMixin, DestroyModelMixin
from rest_framework.generics import GenericAPIView

class BookListCreateMixinView(CreateModelMixin, ListModelMixin, GenericAPIView):
    queryset = Book.objects.all()
    serializer_class = BookModelSerializer

    def get(self, request, *args, **kwargs):
        return self.list(request, *args, **kwargs)

    def post(self, request, *args, **kwargs):
        return self.create(request, *args, **kwargs)

class BookDetailMixinView(RetrieveModelMixin, UpdateModelMixin, DestroyModelMixin, GenericAPIView):
    queryset = Book.objects.all()
    serializer_class = BookModelSerializer

    def get(self, request, *args, **kwargs):
        return self.retrieve(request, *args, **kwargs)

    def put(self, request, *args, **kwargs):
        return self.update(request, *args, **kwargs)

    def delete(self, request, *args, **kwargs):
        return self.destroy(request, *args, **kwargs)
```


## DRF Generic Views

DRF combines `GenericAPIView` and mixins into single, powerful classes:

```python
from rest_framework.generics import ListCreateAPIView, RetrieveUpdateDestroyAPIView

class BookListCreateView(ListCreateAPIView):
    queryset = Book.objects.all()
    serializer_class = BookModelSerializer

class BookRetrieveUpdateDestroyView(RetrieveUpdateDestroyAPIView):
    queryset = Book.objects.all()
    serializer_class = BookModelSerializer
```

```python
# urls.py
urlpatterns = [
    path('books/', BookListCreateView.as_view()),
    path('books/<int:pk>/', BookRetrieveUpdateDestroyView.as_view()),
]
```


## lookup_field and lookup_url_kwarg

By default, DRF looks up objects by `pk`. You can change this to any other field:

```python
class BookBySlugView(RetrieveAPIView):
    queryset = Book.objects.all()
    serializer_class = BookModelSerializer
    lookup_field = 'slug'
    lookup_url_kwarg = 'book_slug'
```

```python
# urls.py
path('books/<slug:book_slug>/', BookBySlugView.as_view()),
```


## Choosing the Right View Type

| Type | Code | Control | Best For |
|------|------|---------|----------|
| `@api_view` | Low | High | Quick demos, simple endpoints |
| `APIView` | Medium | High | Custom logic per method |
| `GenericAPIView` | Medium | Medium | Reuse `get_queryset`, `get_serializer` |
| Mixins + GenericAPIView | Low | Medium | Combine specific behaviors |
| Generic Views | Lowest | Low | Standard CRUD endpoints |


## Summary

- `APIView` separates HTTP methods into `get()`, `post()`, `put()`, `delete()`.
- `GenericAPIView` adds `get_queryset()` and `get_serializer()` helpers.
- Mixins add ready-made actions: `list`, `create`, `retrieve`, `update`, `destroy`.
- Generic views like `ListCreateAPIView` combine a `GenericAPIView` and the appropriate mixins in one class.
- Use `lookup_field` to look up objects by a field other than `pk`.
